# 08 — Queries, Keys, and Attention Scores

**Description:** Project embeddings into queries and keys, compute every query–key dot product, and visualize the resulting relevance scores.
**Level:** Beginner
**Tags:** Language Models, Attention, Queries, Keys, Matrix Multiplication

Notebook 07 showed that contextual representations need a principled way to decide which tokens matter. Attention begins by giving every token two learned descriptions: a **query** for what it seeks and a **key** for what it offers.

By the end, you will be able to:

- compute queries and keys with linear projections;
- interpret a query–key dot product as a compatibility score;
- build the complete score matrix as $QK^T$; and
- track all important shapes.

This notebook deliberately stops at raw scores. Scaling, masking, and softmax belong to Notebook 09.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

np.set_printoptions(precision=3, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")

## 1. Start with a short sequence

Each row is one token embedding. Our sequence length is $T=4$ and embedding width is $d_{model}=3$. The numbers are small and fixed so every operation is inspectable.

In [ ]:
tokens = ["the", "robot", "fixed", "it"]
X = np.array([
    [1.0, 0.0, 0.2],
    [0.2, 1.0, 0.6],
    [0.1, 0.7, 1.0],
    [0.8, 0.2, 0.4],
])

print("X shape:", X.shape)  # (tokens, d_model)
print(X)

## 2. A query describes what a token seeks

A learned matrix $W_Q$ projects each embedding into query space:

$$Q = XW_Q$$

The same matrix is applied to every position. Here $d_k=2$, so each token receives a 2D query.

In [ ]:
W_Q = np.array([
    [1.0, 0.0],
    [0.0, 0.8],
    [0.5, 0.5],
])
Q = X @ W_Q

print("X:  ", X.shape)
print("W_Q:", W_Q.shape)
print("Q:  ", Q.shape)
for token, query in zip(tokens, Q):
    print(f"{token:>5}: {query}")

## 3. A key describes what a token offers

A separate matrix produces keys:

$$K = XW_K$$

Queries and keys have the same final width so their dot products are defined. Their roles differ, so their projection matrices need not be equal.

In [ ]:
W_K = np.array([
    [0.6, 0.2],
    [0.1, 1.0],
    [0.8, 0.3],
])
K = X @ W_K

print("W_K:", W_K.shape)
print("K:  ", K.shape)
for token, key in zip(tokens, K):
    print(f"{token:>5}: {key}")

### Why not compare embeddings directly?

The projections let the model learn a task-specific matching space. The embedding can preserve many features, while $W_Q$ and $W_K$ select and recombine the features useful for routing information in this head.

## 4. One query meets one key

The raw relevance of source position $j$ to target position $i$ is a dot product:

$$s_{ij}=q_i \cdot k_j$$

Rows will represent **query/target** positions. Columns will represent **key/source** positions.

In [ ]:
target_index = tokens.index("it")
source_index = tokens.index("robot")
score = Q[target_index] @ K[source_index]

print("query from 'it':   ", Q[target_index])
print("key from 'robot': ", K[source_index])
print("compatibility:    ", score)
print("manual check:     ", sum(Q[target_index] * K[source_index]))

### Your turn: inspect another pair

Change `source_index` to the position of `fixed`. Does the `it` query match that key more or less strongly? A raw dot product can be positive, zero, or negative; it is not yet a probability.

## 5. Compute every pair at once

Matrix multiplication evaluates all $T^2$ query–key pairs:

$$S = QK^T$$

Shape arithmetic predicts $(T,d_k)(d_k,T)=(T,T)$. Entry `S[i, j]` is exactly `Q[i] @ K[j]`.

In [ ]:
scores = Q @ K.T

print("Q shape:  ", Q.shape)
print("K.T shape:", K.T.shape)
print("S shape:  ", scores.shape)
print(scores)

assert scores.shape == (len(tokens), len(tokens))
assert np.isclose(scores[target_index, source_index], score)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(scores, cmap="coolwarm")
ax.set_xticks(range(len(tokens)), tokens)
ax.set_yticks(range(len(tokens)), tokens)
ax.set(xlabel="Key / source position", ylabel="Query / target position", title=r"Raw attention scores $QK^T$")
for i in range(len(tokens)):
    for j in range(len(tokens)):
        ax.text(j, i, f"{scores[i, j]:.2f}", ha="center", va="center")
fig.colorbar(image, ax=ax, label="dot-product score")
plt.show()

Read one row at a time. The `it` row asks: “How compatible is the query from `it` with the key at each source position?” Columns are possible information sources. Keeping this orientation straight prevents a common implementation error.

## 6. Verify the matrix with loops

The loop version is slower but exposes the meaning of each cell. It should exactly reproduce the matrix multiplication.

In [ ]:
scores_by_loop = np.zeros((len(tokens), len(tokens)))
for i in range(len(tokens)):
    for j in range(len(tokens)):
        scores_by_loop[i, j] = Q[i] @ K[j]

print("largest difference:", np.abs(scores - scores_by_loop).max())
assert np.allclose(scores, scores_by_loop)

## 7. Projections control matching

To see why $W_Q$ and $W_K$ are learnable, perturb one entry of $W_Q$. It changes the queries—and therefore an entire row/column pattern of pairwise scores—without changing the original embeddings.

In [ ]:
W_Q_changed = W_Q.copy()
W_Q_changed[2, 0] += 1.0
Q_changed = X @ W_Q_changed
scores_changed = Q_changed @ K.T

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
for ax, matrix, title in zip(axes, [scores, scores_changed], ["Original scores", "After changing $W_Q$"]):
    im = ax.imshow(matrix, cmap="coolwarm", vmin=min(scores.min(), scores_changed.min()), vmax=max(scores.max(), scores_changed.max()))
    ax.set_xticks(range(len(tokens)), tokens)
    ax.set_yticks(range(len(tokens)), tokens)
    ax.set(xlabel="key", ylabel="query", title=title)
fig.colorbar(im, ax=axes, label="score")
plt.show()

## 8. Shape checklist

For sequence length $T$, model width $d_{model}$, and key width $d_k$:

| Quantity | Shape | Meaning |
| --- | --- | --- |
| $X$ | $(T,d_{model})$ | one embedding per token |
| $W_Q,W_K$ | $(d_{model},d_k)$ | learned projections |
| $Q,K$ | $(T,d_k)$ | one query/key per token |
| $QK^T$ | $(T,T)$ | every target–source score |

The score matrix is square because every token can compare itself with every token, not because $d_k=T$.

## 9. Challenges

1. Compute the score matrix with `np.einsum`.
2. Set $W_Q=W_K$. What symmetry appears in the score matrix, and why?
3. Double every entry of $Q$. What happens to the raw scores?
4. Add a fifth token to `X` and predict every new shape before running the cells.

## Takeaways

- $Q=XW_Q$ describes what each target position seeks.
- $K=XW_K$ describes what each source position offers.
- A dot product measures query–key compatibility.
- $QK^T$ computes all pairwise scores in one matrix multiplication.
- Rows are queries/targets and columns are keys/sources.
- Scores are unrestricted numbers, not probabilities. Notebook 09 will turn them into attention patterns.